In [1]:
import django
import os
from django.db import close_old_connections

os.environ.setdefault("DJANGO_SETTINGS_MODULE", "config.settings.dev")
os.environ["DJANGO_ALLOW_ASYNC_UNSAFE"] = "true"

django.setup()

In [2]:
# !ls -la /tmp/rago/cache
! rm -rf /tmp/rago/cache

In [5]:
%%time
from __future__ import annotations

import os, re, json, time, random, asyncio, logging, textwrap
from pathlib import Path
from hashlib import sha256
from typing import Any, Dict, List, Optional, Tuple, Literal, Set, Callable, Union, cast

from django.conf import settings
from django.db.models import QuerySet

# ---- Models (no writes in this notebook) ----
from literev.models import ProjectRAG, ProjectDocumentRAG, Document

# ---- Rago / LLM utilities you already use ----
from rago import Rago
from rago.retrieval import StringRet
from rago.augmented import OpenAIAug
from rago.generation import OpenAIGen
from rago.extensions.cache import CacheFile

# ---- Faithfulness scorer you already have ----
from literev.libs.scoring import get_faithfulness_score

# ---------- Logging ----------
logger = logging.getLogger("rag_notebook")
logger.setLevel(logging.DEBUG)
if not logger.handlers:
    h = logging.StreamHandler()
    h.setFormatter(logging.Formatter("%(levelname)s:rag_notebook:%(message)s"))
    logger.addHandler(h)

# ---------- Config ----------
ENGINE = "openai"
EMBED_MODEL = "text-embedding-3-small"   # modern embedding model
TOP_N = 10
PERSIST_CACHE = True
USE_LLM_FOR_FAITHFULNESS = True
USE_SENTENCE_CHUNKS = False

CACHE_DIR = Path("/tmp/rago/cache")
CACHE_DIR.mkdir(parents=True, exist_ok=True)
DOC_CACHE   = CacheFile(target_dir=CACHE_DIR / "documents")          # minor/major
AUG_CACHE   = CacheFile(target_dir=CACHE_DIR / f"aug_{ENGINE}")      # retrieval cache
GEN_CACHE   = CacheFile(target_dir=CACHE_DIR / f"gen_{ENGINE}")      # generation cache
FAITH_CACHE = CacheFile(target_dir=CACHE_DIR / f"faith_{ENGINE}")    # faithfulness cache

# =====================================================================
#                1) Colab Minor/Major — EXACT approach
# =====================================================================

# --- optional ftfy fix (best-effort) ---
try:
    from ftfy import fix_text as _ftfy
except Exception:
    _ftfy = None

def _fix_mojibake(s: str) -> str:
    if _ftfy:
        return _ftfy(s)
    rep = {
        "â":"’","â":"–","â":"—","â":"“","â":"”",
        "Ã©":"é","Ã¨":"è","Ãª":"ê","Ã«":"ë","Ã":"à","Ã¢":"â",
        "Ã§":"ç","Ã®":"î","Ã´":"ô","Ã»":"û","Ã¼":"ü","Ã¶":"ö"
    }
    for k,v in rep.items():
        s = s.replace(k,v)
    return s

def normalize_text(raw: str) -> str:
    s = raw or ""
    if "\\'" in s: s = s.replace("\\'", "'")
    if '\\"' in s: s = s.replace('\\"', '"')
    if "\\n" in s: s = s.replace("\\n", "\n")
    if "\\t" in s: s = s.replace("\\t", "\t")
    def _u(m): return chr(int(m.group(1), 16))
    s = re.sub(r"\\u([0-9a-fA-F]{4})", _u, s)
    s = _fix_mojibake(s)
    s = re.sub(r"[ \t]+", " ", s)
    s = re.sub(r"[ \t]*\n[ \t]*", "\n", s).strip()
    return s

############################################################
import re
import unicodedata

def _normalize_text(text: str) -> str:
    # NFKC: normaliza comillas, espacios finos, etc. Mantiene diacríticos (importante en francés).
    t = unicodedata.normalize("NFKC", text)
    # Espacios colapsados
    texto = " ".join(t.split())
    texto = texto.replace('\r\n', '\n').replace('\r', '\n')
    texto = texto.replace('\\n', r'\n')  # desescapar "\n" visibles
    texto = texto.replace(r'\\n', r'\n')  # desescapar "\n" visibles
    texto = texto.replace(u'\\xa0', ' ')
    return texto

def _get_sentences(texto):
    abreviaciones_legales = [
        "art", "Art", "al", "alinéa", "par", "§",
        "M", "Mme", "Mlle", "Dr", "Pr", "Av", "Not", "Mgr", "Me", "St",
        "etc", "cf", "ex", "i.e", "e.g", "op.cit", "loc.cit",
        "c.c", "c.p", "c.trav", "c.comm", "c.consom", "c.pen", "C.deC", "C.fam",
        "No", "n°", "vol", "fig", "p", "pp", "éd"
    ]

    #texto = """
    #Conformément à l’art. 1103 du Code civil, les contrats tiennent lieu de loi à ceux qui les ont faits.
    #Toute personne est tenue d’exécuter les conventions légalement formées.
    #Les juges ne peuvent pas modifier les clauses librement convenues par les parties.
    #Article 1. Les dispositions s’appliquent à tous les contrats.
    #Article 1.1. Les dispositions s’appliquent à tous les contrats.
    #M. Dupont a signé le contrat.
    #"""

    # =========================
    # Split robusto sin look-behind variable
    # =========================
    PLACEHOLDER = "\ue000"

    escaped_abbrs = [re.escape(a) for a in abreviaciones_legales]
    abbr_regex = re.compile(r'\b(?:' + '|'.join(escaped_abbrs) + r')\.', flags=re.UNICODE)

    # 1.1. / 2.3.4. -> proteger el punto final
    enum_multilevel_trailing = re.compile(r'\b(\d+(?:\.\d+)+)\.(?=\s|$)')
    # 1. -> proteger el punto final si después viene mayúscula (típico en “Article 1. Les …”)
    enum_single_trailing = re.compile(r'\b(\d+)\.(?=\s+[A-ZÀ-ÖØ-Ý])')
    # puntos entre dígitos (decimales o internos de enumeraciones)
    decimal_or_internal = re.compile(r'(?<=\d)\.(?=\d)')

    def protect(text: str) -> str:
        # 1) Abreviaciones (M., etc.)
        text = abbr_regex.sub(lambda m: m.group(0)[:-1] + PLACEHOLDER, text)
        # 2) Enumeraciones multi-nivel: proteger el punto final
        text = enum_multilevel_trailing.sub(lambda m: m.group(1) + PLACEHOLDER, text)
        # 3) Enumeración de un nivel (1.) si luego hay mayúscula
        text = enum_single_trailing.sub(lambda m: m.group(1) + PLACEHOLDER, text)
        # 4) Puntos entre dígitos (decimales y puntos internos)
        text = decimal_or_internal.sub(PLACEHOLDER, text)
        return text

    def unprotect(text: str) -> str:
        return text.replace(PLACEHOLDER, '.')

    texto_protegido = protect(texto)

    # Corta tras ., ?, ! seguidos de espacio(s) y mayúscula; también por saltos de línea
    pattern = r'(?<=[.?!])\s+(?=[A-ZÀ-ÖØ-Ý])|\n+'
    oraciones_regex = [unprotect(s.strip()) for s in re.split(pattern, texto_protegido.strip()) if s.strip()]

    # print("=== Sentences ===")
    # for i, o in enumerate(oraciones_regex, 1):
    #     print(f"{i} -- {o}")

    return oraciones_regex
############################################################

# --- FR/CH legal sentence splitter (shield → split → repair) ---
DOT = "⟂"
ABBR = r"(?:art|al|let|consid|cf|n|no|n°|ch|p|etc)"
ABBR_DOT = re.compile(rf"\b({ABBR})\.", re.IGNORECASE)
DOTTED_NUM = re.compile(r"(?<=\d)\.(?=\d)")
PARENS = re.compile(r"\((.*?)\)", re.DOTALL)
SPLIT_RE = re.compile(r"([.!?]+)(?=(?:\s+)(?:[A-ZÉÈÊÀÂÎÔÛÄËÏÖÜÇ0-9\"'»\(\[]))")

def _shield(t: str) -> str:
    t = ABBR_DOT.sub(lambda m: m.group(1) + DOT, t)
    def ps(m):
        inner = m.group(1)
        inner = re.sub(r"\.\s", DOT + " ", inner)
        inner = inner.replace("..", DOT + DOT).replace(".;", DOT + ";")
        return f"({inner})"
    t = PARENS.sub(ps, t)
    t = DOTTED_NUM.sub(DOT, t)
    return t

def _unshield(t: str) -> str:
    return t.replace(DOT, ".")

def split_sentences_fr_legal(text: str) -> List[str]:
    s = _shield(text)
    parts, last = [], 0
    for m in SPLIT_RE.finditer(s):
        end = m.end(1)
        seg = s[last:end].strip()
        if seg: parts.append(seg)
        last = end
    tail = s[last:].strip()
    if tail: parts.append(tail)
    parts = [_unshield(p) for p in parts if p.strip()]

    repaired: List[str] = []
    for p in parts:
        if repaired:
            prev = repaired[-1]
            if re.match(r"^[a-zà-ÿ“”\"'»\)\];,:-]", p):        # attach weak-start
                repaired[-1] = prev + " " + p; continue
            if re.search(r"[;,]$", prev):                       # trailing comma/semicolon
                repaired[-1] = prev + " " + p; continue
            if re.fullmatch(r"\d+\.", prev) and re.match(r"^\d+\.\d+", p):  # 1. with 1.1
                repaired[-1] = prev + " " + p; continue
        repaired.append(p)

    merged: List[str] = []
    for s in repaired:
        if (merged and
            (re.fullmatch(r"\([\s\S]{0,160}\)", s) or re.fullmatch(r"\([\s\S]{0,160}\)\.", s)) and
            re.search(r"\bart\b|\bATF\b|\bACJC\b|\bCPC\b|\bCC\b|\bLTF\b", s, re.IGNORECASE)):
            merged[-1] = merged[-1].rstrip() + " " + s
        else:
            merged.append(s)

    out: List[str] = []
    i = 0
    while i < len(merged):
        cur = merged[i]
        if re.fullmatch(r"[\"']?\d+(?:\.\d+)*\.", cur) and i + 1 < len(merged):
            out.append(cur + " " + merged[i+1]); i += 2
        else:
            out.append(cur); i += 1
    return out

# --- token aware packer (tiktoken optional) ---
try:
    import tiktoken
    _ENC = tiktoken.get_encoding("cl100k_base")
    def _tlen(x: str) -> int: return len(_ENC.encode(x))
    def _tail_tokens(txt: str, keep: int) -> str:
        ids = _ENC.encode(txt)
        keep = max(0, len(ids) - keep)
        return _ENC.decode(ids[keep:])
except Exception:
    _ENC = None
    def _tlen(x: str) -> int: return max(1, len(x)//4)
    def _tail_tokens(txt: str, keep: int) -> str:
        return txt[-keep*4:]

def pack_chunks(
    sents: List[str],
    max_tokens: int = 900,
    overlap_tokens: int = 80,
    min_sent_tokens: int = 5,
) -> List[str]:
    chunks: List[str] = []; cur: List[str] = []; cur_tok = 0
    i = 0
    while i < len(sents):
        s = (sents[i] or "").strip()
        if not s: i += 1; continue
        if _tlen(s) < min_sent_tokens and i + 1 < len(sents):
            s = s + " " + sents[i+1]; i += 1
        sl = _tlen(s)
        if sl > max_tokens:
            # hard-wrap long sentence
            words = s.split(" "); buf=[]; bt=0
            for w in words:
                add = (" " if buf else "") + w; wt = _tlen(add)
                if bt + wt > max_tokens:
                    chunks.append("".join(buf)); buf=[w]; bt=_tlen(w)
                else:
                    buf.append(add); bt += wt
            if buf: chunks.append("".join(buf))
            i += 1; continue
        if cur_tok + sl <= max_tokens:
            cur.append(s); cur_tok += sl; i += 1
        else:
            if cur: chunks.append(" ".join(cur).strip())
            if overlap_tokens and chunks:
                ov = _tail_tokens(chunks[-1], overlap_tokens)
                cur = [ov] if ov else []; cur_tok = _tlen(ov) if ov else 0
            else:
                cur = []; cur_tok = 0
    if cur: chunks.append(" ".join(cur).strip())
    return [c for c in chunks if c]

# ---- Classification schema & validator ----
from pydantic import BaseModel, field_validator

class Classification(BaseModel):
    majeure: List[str]
    mineure: List[str]
    @field_validator("majeure", "mineure", mode="before")
    def _dedup(cls, v):
        return list(dict.fromkeys(v))  # preserve order

def expected_ids(n: int) -> List[str]:
    return [f"CHUNK_{i}" for i in range(n)]

def validate_classification(n: int, c: Classification) -> Tuple[bool, str]:
    exp = set(expected_ids(n))
    got = set(c.majeure) | set(c.mineure)
    errs = []
    unknown = got - exp
    if unknown: errs.append(f"IDs inconnus: {sorted(unknown)}")
    missing = exp - got
    if missing: errs.append(f"IDs manquants: {sorted(missing)}")
    both = set(c.majeure) & set(c.mineure)
    if both: errs.append(f"IDs dans les deux catégories: {sorted(both)}")
    return (len(errs) == 0, "; ".join(errs))

# SYSTEM_PROMPT = (
#     "Tu es juriste. On te fournit des 'chunks' numérotés (CHUNK_i) d'un arrêt. "
#     "Classe CHAQUE chunk en exactement UNE catégorie:\n"
#     "- Majeure: règle de droit générale/applicable (loi, règlement, jurisprudence, standard, test, compétence, recevabilité).\n"
#     "- Mineure: faits/circonstances de l'espèce, application de la règle aux faits, procédure concrète, montants/dates.\n"
#     "Contraintes:\n"
#     "1) JSON UTF-8 strict: {\"majeure\": [...], \"mineure\": [...]}\n"
#     "2) Chaque ID CHUNK_i apparaît UNE seule fois, soit dans majeure, soit dans mineure.\n"
#     "3) N'invente pas d'IDs. Aucune autre clé ni commentaire.\n"
#     "Ambigu: choisis la fonction dominante; hésitation -> 'majeure' si norme/standard clair, sinon 'mineure'."
# )

SYSTEM_PROMPT = (
    "Tu es juriste. On te fournit des 'chunks' numérotés (CHUNK_i) d'un arrêt. "
    "Classe CHAQUE chunk en exactement UNE catégorie:\n"
    "- Majeure: règle de droit générale/applicable (loi, règlement, jurisprudence, standard, test, compétence, recevabilité).\n"
    "- Mineure: faits/circonstances de l'espèce, application de la règle aux faits, procédure concrète, montants/dates.\n"
    "Contraintes STRICTES:\n"
    "1) Sors UNIQUEMENT un JSON UTF-8 de la forme EXACTE: {\"majeure\": [...], \"mineure\": [...]}\n"
    "2) Utilise les clés au SINGULIER: \"majeure\" et \"mineure\" (pas de pluriel, pas d'autres clés).\n"
    "3) Chaque ID CHUNK_i apparaît UNE seule fois: soit dans \"majeure\", soit dans \"mineure\".\n"
    "4) N'invente pas d'IDs, n'ajoute aucun commentaire ni méta.\n"
    "Ambigu: choisis la fonction dominante; hésitation -> 'majeure' si une norme/standard clair est énoncé, sinon 'mineure'."
)

# ---- helper: normalize plural keys if the model returns them ----
def _dbg_preview(s: str, n: int = 400) -> str:
    if s is None:
        return "None"
    s = str(s)
    return (s[:n] + " …[truncated]… " + s[-n:]) if len(s) > 2*n else s

def _normalize_classif_payload(n: int, data: dict) -> dict:
    # 1) normalize keys (singular, lowercase)
    key_map = {"majeures":"majeure","maj":"majeure","majors":"majeure",
               "mineures":"mineure","min":"mineure","minors":"mineure"}
    fixed = {}
    for k, v in data.items():
        kk = key_map.get(k.strip().lower(), k.strip().lower())
        fixed[kk] = v

    # 2) coerce items to proper "CHUNK_i" strings
    def coerce(arr):
        out = []
        for x in (arr or []):
            if isinstance(x, int) or (isinstance(x, str) and x.isdigit()):
                x = f"CHUNK_{int(x)}"          # bare numbers → CHUNK_i
            elif isinstance(x, str) and not x.startswith("CHUNK_") and x.replace("CHUNK","").strip().isdigit():
                # handles 'chunk 12' or 'Chunk_12' weirdness
                x = "CHUNK_" + re.sub(r"\D+", "", x)
            out.append(str(x))
        return out

    if "majeure" in fixed: fixed["majeure"] = coerce(fixed["majeure"])
    if "mineure" in fixed: fixed["mineure"] = coerce(fixed["mineure"])
    return fixed

def build_user_prompt(chunks: List[str]) -> str:
    n = len(chunks)
    lines = [f"Texte à classifier (ne pas modifier le texte). N={n}. "
             f"IDs valides: CHUNK_0 à CHUNK_{n-1}. "
             f"Tu dois renvoyer exactement {n} IDs au total (majeure + mineure = {n}). "
             "Les SEULES clés autorisées sont «majeure» et «mineure» (singulier). "
             "Chaque ID doit apparaître une seule fois et au format exact «CHUNK_i» (chaîne)."]
    for i, t in enumerate(chunks):
        lines.append(f"\nCHUNK_{i}:\n«{t}»")
    lines.append("\nRéponds UNIQUEMENT par le JSON demandé.")
    
    return "\n".join(lines)

# ---- OpenAI JSON caller ----
from openai import OpenAI
_openai_client = OpenAI(api_key=getattr(settings, "OPENAI_API_KEY", ""))


def _classification_schema(n: int) -> dict:
    # "CHUNK_\\d+" strings only, unique, any length up to n (validator will check counts)
    id_schema = {"type": "string", "pattern": r"^CHUNK_\d+$"}
    return {
      "name": "classification_schema",
      "schema": {
        "type": "object",
        "properties": {
          "majeure": {"type": "array", "items": id_schema, "uniqueItems": True},
          "mineure": {"type": "array", "items": id_schema, "uniqueItems": True},
        },
        "required": ["majeure", "mineure"],
        "additionalProperties": False,
      },
      "strict": True
    }


def openai_llm_call(
    system_prompt: str,
    user_prompt: str,
    *,
    model: str = "gpt-4.1-mini",
    temperature: float = 0.0,
    max_tokens: int | None = None,
    json_only: bool = True,
    retries: int = 3,
    base_backoff: float = 0.8,
) -> str:
    for attempt in range(retries + 1):
        try:
            kwargs = dict(
                model=model,
                messages=[
                    {"role": "system", "content": system_prompt},
                    {"role": "user", "content": user_prompt},
                ],
                temperature=temperature,
            )
            if max_tokens is not None:
                kwargs["max_tokens"] = max_tokens
            if json_only:
                kwargs["response_format"] = {"type": "json_object"}

            resp = _openai_client.chat.completions.create(**kwargs)
            return resp.choices[0].message.content or ""

        except Exception as e:
            if attempt >= retries:
                raise
            sleep_s = base_backoff * (2 ** attempt) * (1 + 0.25 * random.random())
            logger.debug(f"[LLM] retry {attempt+1}/{retries} after error: {e} (sleep={sleep_s:.2f}s)")
            time.sleep(sleep_s)

def classify_chunks_llm(chunks: List[str], llm_call: Callable[[str,str], str]) -> Classification:
    sys_p = SYSTEM_PROMPT
    usr_p = build_user_prompt(chunks)

    schema = _classification_schema(len(chunks))
    def _call(u: str) -> str:
        # tiny wrapper to pass schema when possible
        return openai_llm_call(sys_p, u, json_only=False, max_tokens=4096,
                               model="gpt-4.1-mini",  # you already use this
                               response_format_schema=schema)

    logger.debug(
        "[MM] classify_chunks_llm: n_chunks=%d, approx_chars=%d",
        len(chunks), sum(len(c) for c in chunks)
    )

    for attempt in range(3):
        logger.debug("[MM] attempt %d/3 — sending prompt with %d CHUNKs", attempt + 1, len(chunks))
        # raw = llm_call(sys_p, usr_p)
    
        raw = _call(usr_p) if 'response_format_schema' in openai_llm_call.__code__.co_varnames else llm_call(sys_p, usr_p)
        
        if raw is None:
            raw = ""
        raw = raw.strip()

        logger.debug("[MM] attempt %d — raw_len=%d", attempt + 1, len(raw))
        logger.debug("[MM] attempt %d — raw_head_tail=%s", attempt + 1, _dbg_preview(raw, 500))

        # Try to extract a JSON object from the tail if model added extra prose
        m = re.search(r"\{[\s\S]*\}\s*$", raw)
        body = m.group(0) if m else raw
        if not m:
            logger.debug("[MM] attempt %d — no trailing JSON braces found; using full raw", attempt + 1)

        try:
            data = json.loads(body)
            data = json.loads(body)
            data = _normalize_classif_payload(len(chunks), data)
   
            logger.debug("[MM] attempt %d — parsed JSON keys=%s", attempt + 1, sorted(list(data.keys())))

            # Detect common key mistakes without altering behavior
            if ("majeures" in data) or ("mineures" in data):
                logger.debug("[MM][WARN] attempt %d — plural keys detected (expected 'majeure'/'mineure'): %s",
                             attempt + 1, sorted(list(data.keys())))

            # Quick counts before validation
            def _safe_len(k): 
                v = data.get(k, [])
                return len(v) if isinstance(v, list) else f"type={type(v).__name__}"
            logger.debug("[MM] attempt %d — counts: majeure=%s, mineure=%s",
                         attempt + 1, _safe_len("majeure"), _safe_len("mineure"))

            c = Classification(**data)  # may raise if keys wrong types/missing
            
            logger.debug(f"[{d.id}] CLASSIFY n={len(chunks)} start")
            raw_preview = raw[:400].replace("\n"," ")
            logger.debug(f"[{d.id}] model_raw_head={raw_preview}")
            
            logger.debug(f"[{d.id}] parsed_keys={list(data.keys())}")
            logger.debug(f"[{d.id}] counts pre-validate: maj={len(data.get('majeure',[]))} min={len(data.get('mineure',[]))}")
            logger.debug(f"[{d.id}] sample_maj={data.get('majeure',[])[:5]} sample_min={data.get('mineure',[])[:5]}")
            
            ok, why = validate_classification(len(chunks), c)
            if not ok:
                # targeted diagnostics
                exp = set(expected_ids(len(chunks)))
                got = set(c.majeure) | set(c.mineure)
                unknown = sorted(list(got - exp))[:10]
                missing = sorted(list(exp - got))[:10]
                both = sorted(list(set(c.majeure) & set(c.mineure)))[:10]
                logger.debug(f"[{d.id}] VALIDATE FAIL — unknown={len(unknown)} sample={unknown} "
                             f"missing={len(missing)} sample={missing} both={len(both)} sample={both}")

            
            if not ok:
                logger.debug("[MM][VALIDATOR] attempt %d — invalid: %s", attempt + 1, why)

                # Compute diagnostics
                exp = set(expected_ids(len(chunks)))
                got = set(c.majeure) | set(c.mineure)
                unknown = sorted(list(got - exp))
                missing = sorted(list(exp - got))
                both = sorted(list(set(c.majeure) & set(c.mineure)))
                logger.debug("[MM][VALIDATOR] attempt %d — unknown=%d, missing=%d, both=%d",
                             attempt + 1, len(unknown), len(missing), len(both))
                if unknown:
                    logger.debug("[MM][VALIDATOR] attempt %d — unknown(sample)=%s",
                                 attempt + 1, unknown[:15])
                if missing:
                    logger.debug("[MM][VALIDATOR] attempt %d — missing(sample)=%s",
                                 attempt + 1, missing[:15])
                if both:
                    logger.debug("[MM][VALIDATOR] attempt %d — both(sample)=%s",
                                 attempt + 1, both[:15])

                usr_p = build_user_prompt(chunks) + (
                    f"\n\nATTENTION: sortie invalide ({why}). "
                    "Corrige et renvoie UNIQUEMENT le JSON, sans aucun texte supplémentaire."
                )
                continue  # retry

            # Success
            logger.debug("[MM] attempt %d — VALID. majeure=%d, mineure=%d",
                         attempt + 1, len(c.majeure), len(c.mineure))
            return c

        except Exception as e:
            # JSON parse / pydantic errors
            logger.debug("[MM][PARSE] attempt %d — exception=%s", attempt + 1, repr(e))
            logger.debug("[MM][PARSE] attempt %d — body_head_tail=%s", attempt + 1, _dbg_preview(body, 500))

            usr_p = build_user_prompt(chunks) + (
                f"\n\nATTENTION: JSON invalide ({e}). "
                "Renvoie UNIQUEMENT le JSON demandé, sans commentaire."
            )

    # All attempts failed
    logger.debug("[MM] classification failed after retries. Returning error to caller.")
    raise ValueError("Classification JSON invalide après retries.")

# ---- Materialize IDs → text ----
_CHUNK_RX = re.compile(r"^CHUNK_(\d+)$")
def _ids_to_indices(ids: List[str]) -> List[int]:
    out: List[int] = []
    for cid in ids:
        m = _CHUNK_RX.match(cid)
        if not m:
            raise ValueError(f"ID invalide: {cid!r}")
        out.append(int(m.group(1)))
    return out

def materialize_classification(result: Classification, chunks: List[str], order: str = "document") -> Tuple[List[str], List[str]]:
    maj_idx = _ids_to_indices(result.majeure)
    min_idx = _ids_to_indices(result.mineure)
    if order == "document":
        maj_idx.sort(); min_idx.sort()
    elif order != "model":
        raise ValueError("order must be 'document' or 'model'")
    n = len(chunks)
    for i in maj_idx + min_idx:
        if not (0 <= i < n):
            raise IndexError(f"Index hors limites: {i} (0..{n-1})")
    return [chunks[i] for i in maj_idx], [chunks[i] for i in min_idx]

# =====================================================================
#                2) Project/Docs selection (read-only)
# =====================================================================

def get_top10_doc_ids_from_projectrag(project_rag_id: int) -> List[int]:
    # Reuse exactly the docs already attached to this ProjectRAG, in insertion order
    q = (ProjectDocumentRAG.objects
         .filter(project_rag_id=project_rag_id)
         .order_by("id")
         .values_list("document_id", flat=True))
    seen = set(); ordered = []
    for did in q:
        if did not in seen:
            seen.add(did); ordered.append(did)
    return ordered[:TOP_N]

def ordered_docs(doc_ids: List[int]) -> List[Document]:
    docs = list(Document.objects.filter(id__in=doc_ids))
    by_id = {d.id: d for d in docs}
    return [by_id[i] for i in doc_ids if i in by_id]

# =====================================================================
#                3) RAG answering (no DB writes)
# =====================================================================

class RAGAnswerModel(BaseModel):
    answer: str
    highlight: str

class OpenAIAnswerClient:
    def __init__(self, api_key):
        self.api_key = api_key
        self.document_answering_system_prompt = (
            "You are a factual legal assistant. "
            "Always answer in French based solely on the provided context. "
            "If you are absolutely certain the context has no relevant information, "
            'return exactly the string "Réponse non disponible".'
        )
        # NOTE: braces for the JSON example are escaped with double {{ }}
        self.document_answering_user_prompt = (
            'Question: "{query}"\n\n'
            "Context:\n{context}\n\n"
            "Please respond in JSON format with two fields:\n"
            "{{\n"
            '  "answer": "<your concise answer here>",\n'
            '  "highlight": "<up to 5 consecutive sentences from context>"\n'
            "}}\n\n"
            "Rules:\n"
            "- Do not infer or imagine details.\n"
            "- Do not include information that is not explicitly stated in the context.\n"
            "- The highlight must fully justify the answer.\n"
        )

    def get_answer(self, query: str, chunks: List[str]) -> Dict[str, Any]:
        chunks = [c for c in (chunks or []) if isinstance(c, str) and c.strip()]
        if not chunks:
            return {"answer": "Réponse non disponible", "citation": "", "citation_context": []}

        augmented = OpenAIAug(
            api_key=self.api_key,
            top_k=30,
            model_name=EMBED_MODEL,
            cache=AUG_CACHE,
        )
        generation = OpenAIGen(
            api_key=self.api_key,
            model_name="gpt-4o-mini",
            system_message=self.document_answering_system_prompt,
            prompt_template=self.document_answering_user_prompt,
            temperature=0.0,
            output_max_length=16384,
            api_params={"top_p": 0.0, "frequency_penalty": 0.0, "presence_penalty": 0.0},
            structured_output=RAGAnswerModel,
            cache=GEN_CACHE,
        )

        rag = Rago(retrieval=StringRet(chunks), augmented=augmented, generation=generation)
        rag_answer = rag.prompt(query)
        citation_context = rag.logs.get("augmented", {}).get("result", [])

        if isinstance(rag_answer, RAGAnswerModel):
            return {
                "answer": rag_answer.answer.strip(),
                "citation": rag_answer.highlight.strip(),
                "citation_context": citation_context,
            }
        return {"answer": "Error generating response", "citation": "", "citation_context": citation_context}

# =====================================================================
#                4) Cache helpers (minor/major + faithfulness)
# =====================================================================

def mm_cache_key(raw_document_id: str, engine: str = ENGINE) -> str:
    return f"classify_doc_{raw_document_id}_{engine}"

def compute_document_cache_key(query: str, document_id: int) -> str:
    return sha256(f"{query.strip()}::{document_id}".encode()).hexdigest()

def compute_faithfulness_cache_key(query: str, document_id: int, answer: str, context: list[str] | list[dict] | None) -> str:
    payload = {"query": query.strip(), "document_id": document_id, "answer": (answer or "").strip(), "citation_context": context or []}
    return sha256(json.dumps(payload, ensure_ascii=False, sort_keys=True).encode("utf-8")).hexdigest()

# =====================================================================
#                5) Main: run on ProjectRAG top-10
# =====================================================================

# --- Parameters for this run ---
PROJECT_RAG_ID = int(os.environ.get("PRAG_ID", "312"))  # <-- set as needed

pr = ProjectRAG.objects.select_related("project").get(id=PROJECT_RAG_ID)
project_name = pr.project.name
query = pr.query

doc_ids = get_top10_doc_ids_from_projectrag(PROJECT_RAG_ID)
docs = ordered_docs(doc_ids)

logger.debug(f"[ProjectRAG {PROJECT_RAG_ID}] name={project_name!r} query={query!r}")
logger.debug(f"Using {len(docs)} documents: {doc_ids}")

ans_client = OpenAIAnswerClient(getattr(settings, "OPENAI_API_KEY", ""))

doc_results: List[Dict[str, Any]] = []

for d in docs:
    raw_text = (d.raw_document_text or "").strip()

    if USE_SENTENCE_CHUNKS:
        norm = _normalize_text(raw_text)
        sents = _get_sentences(norm)
        chunks = sents  # 1 sentence == 1 chunk
    else:
        norm = normalize_text(raw_text)
        sents = split_sentences_fr_legal(norm)
        chunks = pack_chunks(sents, max_tokens=900, overlap_tokens=80)

    logger.debug(f"[Doc {d.id}] sentences={len(sents)} chunks={len(chunks)}")

    majeures: List[str] = []
    mineures: List[str] = []

    try:
        print("Starting MM Classification...")
        clf = classify_chunks_llm(chunks, openai_llm_call)
        majeures, mineures = materialize_classification(clf, chunks, order="document")
        logger.debug(f"[Doc {d.id}] majeures={len(majeures)} mineures={len(mineures)}")
        # persist to local cache (read-only to DB)
        if PERSIST_CACHE:
            DOC_CACHE.save(mm_cache_key(d.raw_document_id or f"doc_{d.id}"), {
                "raw_document_id": d.raw_document_id or f"doc_{d.id}",
                "results": [{"majeures": majeures, "mineures": mineures}],
                "pipeline": "colab",
                "meta": {"chunks": len(chunks), "sentences": len(sents)},
            })
            logger.debug(f"[CACHE-SAVE] {mm_cache_key(d.raw_document_id or f'doc_{d.id}')} items=1")
        print("MM classification completed successfully!")
    except Exception as e:
        print(f"MM classification failed!!! {e}")
        logger.warning(f"[CLASSIFY] failed raw_id={d.raw_document_id or d.id}: {e}")
        # also save empty to avoid repeated retries (optional)
        if PERSIST_CACHE:
            DOC_CACHE.save(mm_cache_key(d.raw_document_id or f"doc_{d.id}"), {
                "raw_document_id": d.raw_document_id or f"doc_{d.id}",
                "results": [],
                "valid": False,
                "pipeline": "colab",
                "meta": {"chunks": len(chunks), "sentences": len(sents)},
            })
            logger.debug(f"[CACHE-SAVE] {mm_cache_key(d.raw_document_id or f'doc_{d.id}')} items=0")

    # Retrieval fragments: use ONLY mineures; if none, skip answering for this doc
    mineures_clean = [t for t in mineures if isinstance(t, str) and t.strip()]
    if mineures_clean:
        frags = mineures_clean
        logger.debug(f"[Doc {d.id}] fragments_for_answering(mineures-only)={len(frags)}")
        ans = ans_client.get_answer(query, frags)
        skipped_due_to_no_mineures = False
    else:
        frags = []
        ans = {"answer": "Réponse non disponible", "citation": "", "citation_context": []}
        skipped_due_to_no_mineures = True
        logger.debug(f"[Doc {d.id}] skipping answering: no mineures → forcing 'Réponse non disponible'")
    
    logger.debug(f"[Doc {d.id}] answer_len={len(ans.get('answer',''))} | citation_len={len(ans.get('citation',''))}")

    ans = ans_client.get_answer(query, frags)
    logger.debug(f"[Doc {d.id}] answer_len={len(ans.get('answer',''))} | citation_len={len(ans.get('citation',''))}")

    # Faithfulness (cached)
    if USE_LLM_FOR_FAITHFULNESS and (ans.get("answer") or "").strip():
        fkey = compute_faithfulness_cache_key(query, d.id, ans["answer"], ans.get("citation_context", []))
        cached = FAITH_CACHE.load(fkey)
        if cached is not None:
            faith = float(cached.get("score", 0.0))
        else:
            faith = float(asyncio.run(get_faithfulness_score(query, ans["answer"], ans.get("citation_context", []))))
            if PERSIST_CACHE:
                FAITH_CACHE.save(fkey, {"score": faith})
        logger.debug(f"[Doc {d.id}] faithfulness={faith:.3f}")
    else:
        faith = 0.0

    doc_results.append({
        "document_id": d.id,
        "raw_document_id": d.raw_document_id,
        "procedure_type": d.procedure_type,
        "majeures": majeures,
        "mineures": mineures,
        "answer": ans.get("answer", ""),
        "citation": ans.get("citation", ""),
        "citation_context": ans.get("citation_context", []),
        "faithfulness": faith,
    })

logger.debug("Per-document answering complete.")

# =====================================================================
#                6) Summary + considerations (read-only)
# =====================================================================

class SummaryGeneralAnswer(BaseModel):
    summary: str
    considerations: List[str]

def get_summary(query: str, answers: List[str]) -> Dict[str, Any]:
    answers = [a for a in answers if a and a.strip() and a.strip().lower() not in {
        "réponse non disponible", "no content available", "error generating response"
    }]
    if not answers:
        return {"summary": "Résumé non disponible", "considerations": []}

    prompt = (
        "Sur la base de TOUS les extraits de réponses fournis, écris une phrase de synthèse en français. "
        "Ne mentionne pas de noms. Reste strictement sur ce qui est énoncé. "
        "Si aucune information pertinente n'apparaît, retourne exactement: `Résumé non disponible`.\n\n"
        "Question originale: {query}\n\n"
        "Réponses:\n{context}"
    )

    gen = OpenAIGen(
        api_key=getattr(settings, "OPENAI_API_KEY", ""),
        model_name="gpt-4o-mini",
        prompt_template=prompt,
        temperature=0.0,
        output_max_length=2048,
        api_params={"top_p": 0.0, "frequency_penalty": 0.0, "presence_penalty": 0.0},
        structured_output=SummaryGeneralAnswer,
        cache=GEN_CACHE,
    )

    try:
        obj = cast(SummaryGeneralAnswer, gen.generate(query=query, context=answers))
        return {"summary": obj.summary.strip(), "considerations": obj.considerations}
    except Exception as e:
        logger.warning(f"[SUMMARY] failed: {e}")
        return {"summary": "Résumé non disponible", "considerations": []}

valid_answers = [x["answer"].strip() for x in doc_results
                 if x.get("answer","").strip()
                 and x.get("answer","").strip().lower() not in {"réponse non disponible","no content available","error generating response"}]

summary_block = get_summary(query, valid_answers)
logger.debug(f"Summary: {summary_block.get('summary')!r}")
logger.debug(f"Considerations: {summary_block.get('considerations')}")

# =====================================================================
#                7) Final report (pure dict) + optional export
# =====================================================================

run_report = {
    "project_rag_id": PROJECT_RAG_ID,
    "project_name": project_name,
    "engine": ENGINE,
    "query": query,
    "top_doc_ids": doc_ids,
    "documents": doc_results,
    "summary": summary_block.get("summary"),
    "considerations": summary_block.get("considerations", []),
}

# Quick pretty preview without flooding the output
print(json.dumps({
    k: (v if k != "documents" else f"{len(v)} documents")
    for k, v in run_report.items()
}, ensure_ascii=False, indent=2))

# Optional JSON export (no DB changes)
out_path = f"/tmp/mm_rag_paragraph_gen_{PROJECT_RAG_ID}_{ENGINE}.json"
try:
    with open(out_path, "w", encoding="utf-8") as f:
        json.dump(run_report, f, ensure_ascii=False, indent=2)
    print(out_path)
except Exception as e:
    logger.warning(f"Could not write JSON to {out_path}: {e}")


DEBUG:rag_notebook:[ProjectRAG 312] name='divorce' query='divorce et garde des enfants mineurs'
DEBUG:rag_notebook:[ProjectRAG 312] name='divorce' query='divorce et garde des enfants mineurs'
DEBUG:rag_notebook:Using 10 documents: [118594, 118595, 118596, 118597, 118598, 118599, 118600, 118601, 118602, 118603]
DEBUG:rag_notebook:Using 10 documents: [118594, 118595, 118596, 118597, 118598, 118599, 118600, 118601, 118602, 118603]
DEBUG:rag_notebook:[Doc 118594] sentences=266 chunks=20
DEBUG:rag_notebook:[Doc 118594] sentences=266 chunks=20
DEBUG:rag_notebook:[MM] classify_chunks_llm: n_chunks=20, approx_chars=60660
DEBUG:rag_notebook:[MM] classify_chunks_llm: n_chunks=20, approx_chars=60660
DEBUG:rag_notebook:[MM] attempt 1/3 — sending prompt with 20 CHUNKs
DEBUG:rag_notebook:[MM] attempt 1/3 — sending prompt with 20 CHUNKs


Starting MM Classification...


INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
DEBUG:rag_notebook:[MM] attempt 1 — raw_len=256
DEBUG:rag_notebook:[MM] attempt 1 — raw_len=256
DEBUG:rag_notebook:[MM] attempt 1 — raw_head_tail={"majeure": ["CHUNK_6", "CHUNK_7", "CHUNK_8", "CHUNK_9", "CHUNK_10", "CHUNK_11", "CHUNK_12", "CHUNK_13", "CHUNK_14", "CHUNK_15", "CHUNK_16", "CHUNK_17", "CHUNK_18"], "mineure": ["CHUNK_0", "CHUNK_1", "CHUNK_2", "CHUNK_3", "CHUNK_4", "CHUNK_5", "CHUNK_19"]}
DEBUG:rag_notebook:[MM] attempt 1 — raw_head_tail={"majeure": ["CHUNK_6", "CHUNK_7", "CHUNK_8", "CHUNK_9", "CHUNK_10", "CHUNK_11", "CHUNK_12", "CHUNK_13", "CHUNK_14", "CHUNK_15", "CHUNK_16", "CHUNK_17", "CHUNK_18"], "mineure": ["CHUNK_0", "CHUNK_1", "CHUNK_2", "CHUNK_3", "CHUNK_4", "CHUNK_5", "CHUNK_19"]}
DEBUG:rag_notebook:[MM] attempt 1 — parsed JSON keys=['majeure', 'mineure']
DEBUG:rag_notebook:[MM] attempt 1 — parsed JSON keys=['majeure', 'mineure']
DEBUG:rag_notebook:[MM] attempt 1 — counts: maj

MM classification completed successfully!


INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
DEBUG:rag_notebook:[Doc 118594] answer_len=460 | citation_len=621
DEBUG:rag_notebook:[Doc 118594] answer_len=460 | citation_len=621
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
DEBUG:rag_notebook:[Doc 118594] answer_len=374 | citation_len=621
DEBUG:rag_notebook:[Doc 118594] answer_len=374 | citation_len=621
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
DEBUG:rag_notebook:[Doc 118594] faithfulness=0.875
DEBUG:rag_notebook:[Doc 118594] faithfulness=0.875
DEBUG:rag_notebook:[Doc 118595] sentences=138 chunks=11
DEBUG:rag_notebook:[Doc 118595] sentences=13

...

Starting MM Classification...


INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
DEBUG:rag_notebook:[MM] attempt 1 — raw_len=256
DEBUG:rag_notebook:[MM] attempt 1 — raw_len=256
DEBUG:rag_notebook:[MM] attempt 1 — raw_head_tail={"majeure": ["CHUNK_5", "CHUNK_6", "CHUNK_7", "CHUNK_8", "CHUNK_9", "CHUNK_10", "CHUNK_12", "CHUNK_13", "CHUNK_14", "CHUNK_15"], "mineure": ["CHUNK_0", "CHUNK_1", "CHUNK_2", "CHUNK_3", "CHUNK_4", "CHUNK_11", "CHUNK_16", "CHUNK_17", "CHUNK_18", "CHUNK_19"]}
DEBUG:rag_notebook:[MM] attempt 1 — raw_head_tail={"majeure": ["CHUNK_5", "CHUNK_6", "CHUNK_7", "CHUNK_8", "CHUNK_9", "CHUNK_10", "CHUNK_12", "CHUNK_13", "CHUNK_14", "CHUNK_15"], "mineure": ["CHUNK_0", "CHUNK_1", "CHUNK_2", "CHUNK_3", "CHUNK_4", "CHUNK_11", "CHUNK_16", "CHUNK_17", "CHUNK_18", "CHUNK_19"]}
DEBUG:rag_notebook:[MM] attempt 1 — parsed JSON keys=['majeure', 'mineure']
DEBUG:rag_notebook:[MM] attempt 1 — parsed JSON keys=['majeure', 'mineure']
DEBUG:rag_notebook:[MM] attempt 1 — counts: maj

MM classification completed successfully!


INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
DEBUG:rag_notebook:[Doc 118603] answer_len=99 | citation_len=458
DEBUG:rag_notebook:[Doc 118603] answer_len=99 | citation_len=458
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
DEBUG:rag_notebook:[Doc 118603] answer_len=99 | citation_len=458
DEBUG:rag_notebook:[Doc 118603] answer_len=99 | citation_len=458
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
DEBUG:rag_notebook:[Doc 118603] faithfulness=1.000
DEBUG:rag_notebook:[Doc 118603] faithfulness=1.000
DEBUG:rag_notebook:Per-document answering complete.
DEBUG:rag_notebook:Per-document answering complete.
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 20

{
  "project_rag_id": 312,
  "project_name": "divorce",
  "engine": "openai",
  "query": "divorce et garde des enfants mineurs",
  "top_doc_ids": [
    118594,
    118595,
    118596,
    118597,
    118598,
    118599,
    118600,
    118601,
    118602,
    118603
  ],
  "documents": "10 documents",
  "summary": "Le tribunal a statué sur la garde des enfants mineurs, attribuant celle-ci alternativement au père ou à la mère selon les cas, tout en maintenant l'autorité parentale conjointe ou en réservant des droits de visite spécifiques à l'autre parent, en tenant compte de l'intérêt des enfants et des circonstances familiales.",
  "considerations": [
    "La garde a été attribuée au père ou à la mère selon les décisions judiciaires.",
    "L'autorité parentale conjointe a été maintenue dans plusieurs cas.",
    "Des droits de visite ont été réservés à l'autre parent, souvent limités ou élargis selon les situations.",
    "Les décisions ont pris en compte l'intérêt des enfants et leur 

In [6]:
!cp -rp /tmp/mm_rag_paragraph_gen_312_openai.json ./data/mm_rag_paragraph_gen_312_openai.json
